# DeepONet Burgers — Google Colab 실행 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Marcellinus0602/deeponet-pde-research/blob/main/notebooks/colab_burgers.ipynb)

**위에서부터 한 셀씩 차례로 실행**하면 됩니다 (또는 메뉴 `런타임 → 모두 실행`).
처음 한 번만: 메뉴 `런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU` 선택.

프로젝트 전체(코드·데이터·결과)를 Google Drive 에 두고 씁니다. 세션이 끝나도 전부 Drive 에 남습니다.

```
MyDrive/deeponet-pde-research/        ← 3번 셀이 처음 한 번 GitHub 에서 복사
├── core/  models/  utils/            ← 코드
├── experiments/                      ← 학습 스크립트 (여기서 고쳐서 실행)
├── notebooks/colab_burgers.ipynb     ← 이 노트북
├── data/burgers/                     ← 학습 데이터 저장본 (.pt, 첫 실행 때 자동 생성)
└── runs/                             ← 실험 결과 (자동 생성)
    ├── burgers_results.csv           ← 경우별 test 결과 비교표
    └── burgers_{sin|grf}_{upwind|conservative}/exp_날짜_시간/
        ├── best_model.pth, config.json, history.json
        └── loss_curve.png, burgers_prediction.png
```

- **다음부터 여는 법**: drive.google.com → `deeponet-pde-research/notebooks/colab_burgers.ipynb` 더블클릭
- **코드 고치는 법**: 왼쪽 📁 패널 → `drive/MyDrive/deeponet-pde-research/experiments/...` 더블클릭 → 수정 → `Ctrl+S` (Drive 에 바로 저장)
- **데이터 생성 시간 아끼기**: 로컬에 이미 만든 `data/burgers/*.pt` 가 있으면 Drive 의 `data/burgers/` 에 올려 두면 그대로 불러옴

In [ ]:
# 1. GPU 확인 — "GPU: Tesla T4" 처럼 나오면 OK
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU 없음 → 메뉴 '런타임 → 런타임 유형 변경 → T4 GPU' 선택 후 이 셀부터 다시 실행 (CPU 로도 돌아가지만 학습이 매우 느림)")

In [ ]:
# 2. Google Drive 연결 — 처음엔 접근 허용 창이 뜸 (본인 계정 선택 → 허용)
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 3. 프로젝트 준비 — Drive 에 프로젝트가 없을 때만 GitHub 에서 복사 (있으면 그대로 사용, Drive 에서 고친 코드는 덮어쓰지 않음)
from pathlib import Path

REPO_URL = "https://github.com/Marcellinus0602/deeponet-pde-research.git"
BRANCH = "main"                                                       # 처음 복사할 때 받을 브랜치
PROJECT_DIR = Path("/content/drive/MyDrive/deeponet-pde-research")    # 프로젝트 전체(코드·데이터·결과)가 있는 Drive 폴더

if (PROJECT_DIR / "experiments").exists():
    print("Drive 의 프로젝트를 그대로 사용:", PROJECT_DIR)
else:
    print("처음 실행 → GitHub 에서 Drive 로 프로젝트 복사 중...")
    !git clone -q -b {BRANCH} {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}

## 4. 학습 실행

- **처음 실행하는 경우**는 학습 데이터(수치해)부터 만듭니다. 로그에 `[Compute] ... 신규 생성 중` 이 보이고, numpy(CPU) 계산이라 경우당 5~10분 정도 걸립니다. 만든 데이터는 Drive 의 `data/burgers/` 에 저장됩니다.
- **두 번째부터**는 `[SSD Cache Load]` 가 뜨면서 Drive 에서 바로 불러옵니다.
- 학습(8000 epoch)은 GPU 에서 경우당 몇 분 정도 걸립니다 (로컬 CPU 에서는 약 38분 걸렸음).
- 돌리지 않을 경우는 아래 `CASES` 에서 그 줄 앞에 `#` 을 붙여 빼면 됩니다.
- 학습 설정(epochs, 학습률 등)은 `experiments/train_burgers_{경우}.py` 위쪽 `config` 에서 고치고 `Ctrl+S` → 이 셀 다시 실행.

In [ ]:
# 4. 학습 — 경우마다 experiments/train_burgers_{경우}.py 를 차례로 실행
CASES = [
    "sin_upwind",        # 1) 기존 데이터셋(sin) + 비보존형 솔버
    "sin_conservative",  # 2) 기존 데이터셋(sin) + 보존형 솔버
    "grf_upwind",        # 3) GRF 데이터셋 + 비보존형 솔버
    "grf_conservative",  # 4) GRF 데이터셋 + 보존형 솔버
]

for case in CASES:
    print(f"\n########## {case} ##########", flush=True)
    # -u: 로그를 바로바로 출력 / MPLBACKEND=Agg: 그림은 화면 대신 파일로만 저장 (6번 셀에서 표시)
    # 끝에 "FigureCanvasAgg is non-interactive" 경고가 나와도 무시 (그림 파일은 정상 저장됨)
    !MPLBACKEND=Agg python -u experiments/train_burgers_{case}.py

In [ ]:
# 5. 결과 비교표 — 실행할 때마다 한 줄씩 쌓임 (같은 경우를 여러 번 돌렸으면 아래 줄이 최신)
import pandas as pd

results_path = PROJECT_DIR / "runs" / "burgers_results.csv"
if results_path.exists():
    display(pd.read_csv(results_path))
else:
    print("아직 결과가 없음 → 4번 셀을 먼저 실행")

In [ ]:
# 6. 경우별 가장 최근 실행(끝까지 완료된 것)의 그림 보기
from IPython.display import Image

for case in CASES:
    exps = sorted(p for p in (PROJECT_DIR / "runs" / f"burgers_{case}").glob("exp_*") if (p / "config.json").exists())
    if not exps:
        print(f"{case}: 완료된 실행 없음")
        continue
    latest = exps[-1]
    print(f"\n========== {case} ({latest.name}) ==========")
    for name in ["loss_curve.png", "burgers_prediction.png"]:
        if (latest / name).exists():
            display(Image(filename=str(latest / name), width=900))

## 7. (선택) 다 끝났으면

결과는 이미 Drive 에 저장돼 있습니다. GPU 를 붙잡고 있는 동안 무료 사용 한도(유료는 컴퓨팅 단위)가 계속 줄어드니, 끝났으면 런타임을 끊어 두세요.
아래 셀의 `#` 을 지우고 실행하거나, 메뉴 `런타임 → 런타임 연결 해제 및 삭제`.

**자주 겪는 문제**
- 세션이 끊김 (무료는 최대 약 12시간, 오래 자리를 비우면 더 빨리 끊김): Drive 의 데이터·`best_model.pth` 는 남아 있으니 1~3번 셀을 다시 실행하고, `CASES` 에서 이미 끝난 경우를 빼고 이어서 실행.
- "GPU 를 사용할 수 없음": 무료 GPU 한도 소진 → 몇 시간 뒤 다시 시도 (CPU 로도 돌아가지만 느림).

In [ ]:
# drive.flush_and_unmount()          # Drive 에 아직 안 올라간 파일까지 확실히 저장
# from google.colab import runtime
# runtime.unassign()                 # 런타임 종료 (GPU 반납)